# Exercise 3 — Bright hosts and frame times

*Covers notebooks 04. Difficulty: ★★☆☆☆*

The best coronagraph targets are the brightest stars in the sky, and those saturate. How short must the frames be, and what does that cost?

In [1]:
import warnings
import numpy as np
import astropy.units as u
import matplotlib.pyplot as plt
import esc_etc
from esc_etc import Simulation, use_lazuli_style

use_lazuli_style()
warnings.filterwarnings("ignore", module="synphot")

def frame_for(sim, max_frame=1000 * u.s):
    """Frame time to use: max_frame, or half the saturation time if that is shorter."""
    return min(u.Quantity(max_frame, u.s), 0.5 * sim.saturation_time())


def hours_to_snr(sim, snr=5, max_frame=1000 * u.s):
    """Integration time (hours) to reach snr with the current configuration; NaN above the speckle plateau."""
    return sim.get_int_time_for_snr(snr, frame_for(sim, max_frame))["int_time_continuous"][0].to_value(u.hr)

### 3.0 — A broken cell

The cell below is broken on purpose. Run it, paste the traceback into Jupyter AI (or read it
yourself), fix it, and keep the fixed version: it defines the helper the rest of the exercise uses.

In [2]:
def safe_frame(sim, max_frame=1000 * u.s):
    """Frame time: max_frame, or half the saturation time if that is shorter."""
    # fixed: saturation_time() is an astropy Quantity in seconds, so max_frame must be one too
    return min(u.Quantity(max_frame, u.s), 0.5 * sim.saturation_time())

sim = Simulation.from_default(filter="R3", hoststarmag=-1.4, hoststar_band="johnson_v", planetdeltamag=20, contrast=1e-8, ppgain=10, separation=0.4)
print(safe_frame(sim))

1000.0 s


### 3.1 — Sirius, Pollux and a V = 5 star

For hosts of V = -1.4, 1.1 and 5.0 with a $10^{-8}$ planet at 0.4", tabulate the saturation time,
the frame time from `safe_frame`, the S/N after one hour and the time to S/N 5.

In [3]:
print(f"{'V':>5} {'sat. time':>12} {'frame':>9} {'S/N 1 h':>8} {'t(S/N 5)':>9}")
for V in (-1.4, 1.1, 5.0):
    sim.update(scene__star__mag=V)
    ft = safe_frame(sim)
    print(f"{V:5.1f} {sim.saturation_time():12.0f} {ft:9.1f} {sim.get_snr(3600, ft)['snr']:8.2f} {sim.get_int_time_for_snr(5, ft)['int_time_continuous'][0].to_value(u.hr):9.2f} h")
sim.reset()

    V    sat. time     frame  S/N 1 h  t(S/N 5)


 -1.4         3716 s    1000.0 s     9.99      0.00 h


  1.1        36742 s    1000.0 s     9.89      0.01 h


  5.0       922923 s    1000.0 s     6.09      0.55 h


> ✏️ **Before you run the next cell:** for the V = 5 host, how much S/N is lost in one hour by using 10 s frames instead of 1000 s: 5 %, 30 % or a factor 3? Write your guess down (or put it in the room poll), then run.

### 3.2 — The cost of short frames

For the V = 5 host, S/N in one hour with 1, 10, 100 and 1000 s frames. At which frame time does the
read noise fall below 10 % of the total variance? (Reconstruct the read term as in notebook 04:
$n_\mathrm{pix}\,\sigma_\mathrm{read}^2\,t/t_\mathrm{frame}$.)

In [4]:
sim.update(scene__star__mag=5.0)
s = sim.sensor
for ft in (1, 10, 100, 1000):
    r = sim.get_snr(3600, ft)
    read = (s.read_noise ** 2 * s.num_psf_pixels * (3600 * u.s / (ft * u.s))).to_value(u.electron)
    print(f"frame {ft:5d} s: S/N {r['snr']:.2f}; read noise {read / r['noise_e'] ** 2:6.1%} of the variance")
sim.reset()

frame     1 s: S/N 0.50; read noise  99.4% of the variance
frame    10 s: S/N 1.56; read noise  94.5% of the variance


frame   100 s: S/N 4.03; read noise  63.3% of the variance


frame  1000 s: S/N 6.09; read noise  14.5% of the variance


### 3.3 — Sirius for real

For Sirius the ETC's saturation time is the planet-aperture estimate. Suppose the host halo limits
frames to 1 s. S/N after one hour, and time to S/N 5, for a $10^{-8}$ planet at 0.4"?

In [5]:
sim.update(scene__star__mag=-1.4)
print(f"1 s frames: S/N in 1 h = {sim.get_snr(3600, 1)['snr']:.2f}, time to S/N 5 = {sim.get_int_time_for_snr(5, 1)['int_time_continuous'][0].to(u.min):.1f}")
sim.reset()

1 s frames: S/N in 1 h = 9.97, time to S/N 5 = 0.1 min


> 🤖 **Ask Jupyter AI:** *The ETC's saturation_time() only models the planet location at the dark-hole contrast (esc-etc issue #33). Using sim.count_rate_components().host, sim.sensor.peak_pixel_fraction and the mask throughput at the separation, estimate the frame time at which the host's own residual halo saturates a pixel, and compare with saturation_time().*
>
> Ask it to state what it assumes about the halo profile outside the dark hole.

***
## Share-back and feedback

Bring back **one plot and one number** to the share-back board, and the observing decision you draw
from them.

> 🤖 **Ask Jupyter AI:** *Draft a GitHub issue for schmidt-observatories/esc-etc describing one thing I
> found missing, confusing or wrong in this exercise, with a minimal code example.*